In [ ]:
!pip install ultralytics grad-cam
# !pip install pytorch-grad-cam

In [ ]:
import os, random
from pathlib import Path
from collections import defaultdict

import yaml


IMG_DIR = Path("/home/smile/work/pbs06/test/images")
LBL_DIR = IMG_DIR.with_name("labels")

# 1) 라벨별 후보 수집  {cls_id: [(img_path, (xc,yc,w,h)), …]}
class_map = defaultdict(list)
for lbl in LBL_DIR.glob("*.txt"):
    img_path = IMG_DIR / f"{lbl.stem}.jpg"
    if not img_path.exists():
        continue
    with open(lbl) as f:
        for ln in f:
            cls, xc, yc, bw, bh = ln.split()
            class_map[int(cls)].append((str(img_path),
                                        tuple(map(float, (xc, yc, bw, bh)))))

# 2) 오름차순 정렬 → 라벨별 20개 무작위 추출
SAMPLES   = 20
img_files = []   # [(img_path, cls_id, (xc,yc,w,h)), …]

for cid in sorted(class_map.keys()):          # ← 오름차순
    random.shuffle(class_map[cid])
    for img_path, box in class_map[cid][:SAMPLES]:
        img_files.append((img_path, cid, box))

print(f"총 샘플: {len(img_files)}")

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import torch, torch.nn as nn
import cv2, numpy as np, matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image

# ────────── 1. 기본 설정 ──────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_path = "/home/smile/work/pbs/model/yolov5/runsbackup_v10/250417_U2L_001_v10/blood_cell8/weights/best.pt"
class_names = {
    0: 'BandNeutrophil',
    1: 'Basophil',
    2: 'Blast',
    3: 'Echinocyte',
    4: 'Elliptocyte',
    5: 'Eosinophil',
    6: 'GiantPlt',
    7: 'Lymphocyte',
    8: 'Monocyte',
    9: 'Myelocyte',
    10: 'Nucleated',
    11: 'PLT',
    12: 'RBC',
    13: 'Reticulocyte',
    14: 'Schistocyte',
    15: 'SegNeutrophil',
    16: 'Smudge',
    17: 'Stomatocyte',
    18: 'TargetCell',
    19: 'TearDropCell',
    20: 'ToxicGranule',
    21: 'ToxicVacuole',
    22: 'clump-PLT',
    23: 'degeneWBC',
    24: 'hyperSeg.'
}

# ────────── 2. 백본 래퍼 ──────────
class YOLOv10Backbone(nn.Module):
    def __init__(self, full, stop_idx=23, target_idx=14):
        super().__init__()
        self.layers = full.model[:stop_idx]
        self.target_idx = target_idx

    def forward(self, x):
        inter = []
        cam_feat = None
        for i, m in enumerate(self.layers):
            if hasattr(m, "f") and m.f not in (None, -1):
                refs = m.f if isinstance(m.f, (list, tuple)) else [m.f]
                x = m([inter[k if k >= 0 else i + k] for k in refs])
            else:
                x = m(x)
            inter.append(x)
            if i == self.target_idx:
                cam_feat = x
        return x, cam_feat

# ────────── 3. 박스 기반 CAM 타겟 ──────────
class YOLOBoxTarget:
    def __init__(self, box, img_size=640, top_k=8):
        self.box = box
        self.img_size = img_size
        self.top_k = top_k

    def __call__(self, activations):
        _, C, H, W = activations.shape
        x1, y1, x2, y2 = self.box
        x1 = int(x1 / self.img_size * W)
        x2 = int(x2 / self.img_size * W)
        y1 = int(y1 / self.img_size * H)
        y2 = int(y2 / self.img_size * H)
        x1, y1 = max(x1, 0), max(y1, 0)
        x2, y2 = min(x2, W), min(y2, H)

        if x1 >= x2 or y1 >= y2:
            return torch.tensor(0.0, device=activations.device)

        patch = activations[0, :, y1:y2, x1:x2]
        scores = patch.abs().mean((1, 2))
        topk = scores.topk(min(self.top_k, C))
        return patch[topk.indices].mean()

# ────────── 4. 모델 초기화 ──────────
full = YOLO(model_path).model.cpu().eval()
backbone = YOLOv10Backbone(full).to(device).eval()
target_layers = [backbone.layers[12]]
cam = GradCAMPlusPlus(model=backbone, target_layers=target_layers)
yolo_model = YOLO(model_path)

# ────────── 5. 이미지 처리 ──────────
img_dir = Path("/home/smile/work/pbs06/test/images")  # 수정: 이미지 폴더 경로
img_files = sorted(img_dir.glob("*.jpg"))
TOP_N = 20

# IMG_DIR = Path("/home/smile/work/pbs06/test/images")
# LBL_DIR = IMG_DIR.with_name("labels")

for img_path in img_files[:TOP_N]:
    img = cv2.imread(str(img_path))
    img = cv2.resize(img, (640, 640))
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    tensor = torch.from_numpy(rgb.transpose(2, 0, 1)).unsqueeze(0).to(device).requires_grad_(True)

    pred = yolo_model(str(img_path), verbose=False)[0]
    if len(pred.boxes) == 0:
        continue

    num_boxes = len(pred.boxes)
    plt.figure(figsize=(5 * num_boxes, 5))
    shown = 0

    for i in range(num_boxes):
        conf = float(pred.boxes.conf[i])
        label = int(pred.boxes.cls[i])
        if conf < 0.2:
            continue

        box = pred.boxes.xyxy[i].cpu().tolist()
        target = [YOLOBoxTarget(box, img_size=640)]

        cam_map = cam(tensor, targets=target, eigen_smooth=False, aug_smooth=False)[0]
        cam_norm = (cam_map - cam_map.min()) / (cam_map.ptp() + 1e-8)
        cam_norm = np.power(np.clip(cam_norm, 0, 1), 1.5)
        overlay = show_cam_on_image(rgb, cam_norm, use_rgb=True, image_weight=0.5)

        plt.subplot(1, num_boxes, shown + 1)
        plt.imshow(overlay)
        plt.axis("off")
        plt.title(f"{class_names.get(label, label)}\nconf={conf:.2f}")
        shown += 1

    plt.suptitle(img_path.name)
    plt.tight_layout()
    plt.show()



In [ ]:
for img_path in img_files[:TOP_N]:
    print(img_path)

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import torch, torch.nn as nn
import cv2, numpy as np, matplotlib.pyplot as plt
# from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam import (
    GradCAM,
    GradCAMPlusPlus,
    ScoreCAM,
    AblationCAM,
    XGradCAM,
    EigenCAM,
    EigenGradCAM,
    LayerCAM,
    FullGrad
)
from pytorch_grad_cam.utils.image import show_cam_on_image

# ────────── 1. YOLOv10 Detect 전 백본 래퍼 ──────────
class YOLOv10Backbone(nn.Module):
    def __init__(self, full, stop_idx=23, target_idx=12):
        super().__init__()
        self.layers = full.model[:stop_idx]
        self.target_idx = target_idx  # CAM 타겟 레이어 인덱스

    def forward(self, x):
        inter = []
        cam_feat = None
        for i, m in enumerate(self.layers):
            if hasattr(m, "f") and m.f not in (None, -1):
                refs = m.f if isinstance(m.f, (list, tuple)) else [m.f]
                x = m([inter[k if k >= 0 else i + k] for k in refs])
            else:
                x = m(x)
            inter.append(x)
            if i == self.target_idx:
                cam_feat = x
        return x, cam_feat

# ────────── 2. 모델 및 백본 초기화 ──────────
model_path = "/home/smile/work/pbs/model/yolov5/runsbackup_v10/250417_U2L_001_v10/blood_cell8/weights/best.pt"
full = YOLO(model_path).model.cpu().eval()
backbone = YOLOv10Backbone(full, stop_idx=23, target_idx=14).eval()
target_layers = [backbone.layers[12]]

# ────────── 3. GradCAM++ 객체 생성 ──────────
cam = EigenCAM(model=backbone, target_layers=target_layers)
# cam = GradCAMPlusPlus(model=backbone, target_layers=target_layers)
# cam = AblationCAM(model=backbone, target_layers=target_layers)
# ────────── 4. 다중 채널 Target 클래스 ──────────
# class MultiChannelTarget:
#     def __init__(self, ch_list):
#         self.ch_list = ch_list

#     def __call__(self, feat):
#         C = feat.shape[1]
#         safe_chs = [ch for ch in self.ch_list if ch < C]
#         if not safe_chs:
#             return torch.tensor(0.0, device=feat.device)
#         return torch.stack([feat[0, ch].mean() for ch in safe_chs]).mean()

    
# class WeightedChannelTarget:
#     def __init__(self, ch_list, weights):
#         self.ch_list = ch_list
#         self.weights = weights

#     def __call__(self, feat):
#         C = feat.shape[1]
#         values = []
#         for ch, w in zip(self.ch_list, self.weights):
#             if ch < C:
#                 values.append(w * feat[0, ch].mean())
#         return torch.stack(values).sum() if values else torch.tensor(0.0, device=feat.device)
    
    
class MultiChannelTarget:
    def __init__(self, ch_list):
        self.ch_list = ch_list

    def __call__(self, feat):
        return torch.stack([feat[0, ch].mean() for ch in self.ch_list]).mean()
    
# ────────── 5. 이미지 루프 (CAM++ 시각화만) ──────────
TOP_K = 1 # 사용할 채널 수

for img_path, *_ in img_files[:15]:  # 앞에서 6장만 예시
    img = cv2.imread(img_path)
    img = cv2.resize(img, (640, 640))
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    tensor = torch.from_numpy(rgb.transpose(2, 0, 1)).unsqueeze(0).requires_grad_(True)

    with torch.no_grad():
        _ = backbone(tensor)  # hook 등록용

    # 2. CAM 내부의 activation 가져오기
    hook = cam.activations_and_grads.activations
    if not hook or not hook[0].shape[1]:
        continue  # skip if no activation
    
    feat = hook[0]  # [1, C, H, W]
    mean = feat[0].abs().mean((1, 2))
    median = feat[0].abs().view(feat.shape[1], -1).median(dim=1).values
    scores = mean - median

    
    
    # scores = feat[0].abs().mean((1, 2))
    # scores = feat[0].abs().mean((1, 2)) / (feat[0].std((1, 2)) + 1e-6)
    topk = scores.topk(min(TOP_K, scores.shape[0]))
    ch_ids = topk.indices.tolist()

    # CAM 타겟 생성 및 계산
    target = MultiChannelTarget(ch_ids)
    # target = WeightedChannelTarget(ch_ids)
    
    cam_map = cam(tensor, targets=[target], eigen_smooth=False, aug_smooth=False)[0]
    # 시각화
    cam_norm = (cam_map - cam_map.min()) / (cam_map.ptp() + 1e-8)
    cam_norm = np.clip(cam_norm, 0, 1)
    cam_norm = np.power(cam_norm, 1.5)  # 2.0 이상으로 올릴수록 고값 강조됨
    overlay = show_cam_on_image(rgb, cam_norm, use_rgb=True, image_weight=0.5)

    plt.figure(figsize=(5, 5))
    plt.imshow(overlay)
    plt.axis("off")
    # plt.title(f"{Path(img_path).name} | chs {ch_ids} | CAM++")
    plt.show()


In [ ]:
from pathlib import Path
from ultralytics import YOLO
import torch, torch.nn as nn
import cv2, numpy as np, matplotlib.pyplot as plt

from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ────────── 1. YOLOv10 Detect 전 백본 래퍼 ──────────
class YOLOv10Backbone(nn.Module):
    def __init__(self, full, stop_idx=23, target_idx=12):
        super().__init__()
        self.layers = full.model[:stop_idx]
        self.target_idx = target_idx

    def forward(self, x):
        inter = []
        cam_feat = None
        for i, m in enumerate(self.layers):
            if hasattr(m, "f") and m.f not in (None, -1):
                refs = m.f if isinstance(m.f, (list, tuple)) else [m.f]
                x = m([inter[k if k >= 0 else i + k] for k in refs])
            else:
                x = m(x)
            inter.append(x)
            if i == self.target_idx:
                cam_feat = x
        return x, cam_feat

# ────────── 2. 사용자 정의 YOLO 타겟 클래스 ──────────
class YOLOBoxTarget:
    def __init__(self, box, img_size=640):
        self.box = box  # [x1, y1, x2, y2]
        self.img_size = img_size

    def __call__(self, activations):  # [1, C, H, W]
        _, C, H, W = activations.shape
        x1, y1, x2, y2 = self.box
        x1 = int(x1 / self.img_size * W)
        x2 = int(x2 / self.img_size * W)
        y1 = int(y1 / self.img_size * H)
        y2 = int(y2 / self.img_size * H)
        x1, y1 = max(x1, 0), max(y1, 0)
        x2, y2 = min(x2, W), min(y2, H)

        if x1 >= x2 or y1 >= y2:
            return torch.tensor(0.0, device=activations.device)

        patch = activations[0, :, y1:y2, x1:x2]
        return patch.mean()

# ────────── 3. 모델 초기화 ──────────
model_path = "/home/smile/work/pbs/model/yolov5/runsbackup_v10/250417_U2L_001_v10/blood_cell8/weights/best.pt"
full = YOLO(model_path).model.cpu().eval()
backbone = YOLOv10Backbone(full, stop_idx=23, target_idx=14).to(device).eval()
# backbone = YOLOv10Backbone(full, stop_idx=23, target_idx=12).eval()
target_layers = [backbone.layers[12]]
cam = GradCAMPlusPlus(model=backbone, target_layers=target_layers)
yolo_model = YOLO(model_path)

# img_files는 [(img_path, class_id, box), ...] 구조라고 가정
TOP_N = 15  # 최대 이미지 수
# for img_path, *_ in img_files[:TOP_N]:
for img_path in img_files[:TOP_N]:
    img = cv2.imread(str(img_path))
    img = cv2.resize(img, (640, 640))
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    tensor = torch.from_numpy(rgb.transpose(2, 0, 1)).unsqueeze(0).requires_grad_(True)

    # YOLO 예측
    pred = yolo_model(img_path, verbose=False)[0]
    if len(pred.boxes) == 0:
        continue

    box = pred.boxes.xyxy[1].cpu().tolist()  # 가장 높은 confidence 박스
    label = int(pred.boxes.cls[0])
    print(label)
    target = [YOLOBoxTarget(box, img_size=640)]

    # CAM 계산
    cam_map = cam(tensor, targets=target, eigen_smooth=False, aug_smooth=False)[0]
    cam_norm = (cam_map - cam_map.min()) / (cam_map.ptp() + 1e-8)
    cam_norm = np.power(np.clip(cam_norm, 0, 1), 1.5)
    overlay = show_cam_on_image(rgb, cam_norm, use_rgb=True, image_weight=0.5)

    # 시각화
    plt.figure(figsize=(5, 5))
    plt.imshow(overlay)
    plt.axis("off")
    plt.title(f"{Path(img_path).name} | class {label}")
    plt.show()


In [ ]:

import os, random
from pathlib import Path
from collections import defaultdict

import yaml


IMG_DIR = Path("/home/smile/work/pbs06/test/images")
LBL_DIR = IMG_DIR.with_name("labels")

# 1) 라벨별 후보 수집  {cls_id: [(img_path, (xc,yc,w,h)), …]}
class_map = defaultdict(list)
for lbl in LBL_DIR.glob("*.txt"):
    img_path = IMG_DIR / f"{lbl.stem}.jpg"
    if not img_path.exists():
        continue
    with open(lbl) as f:
        for ln in f:
            cls, xc, yc, bw, bh = ln.split()
            class_map[int(cls)].append((str(img_path),
                                        tuple(map(float, (xc, yc, bw, bh)))))

# 2) 오름차순 정렬 → 라벨별 20개 무작위 추출
SAMPLES   = 20
img_files = []   # [(img_path, cls_id, (xc,yc,w,h)), …]

for cid in sorted(class_map.keys()):          # ← 오름차순
    random.shuffle(class_map[cid])
    for img_path, box in class_map[cid][:SAMPLES]:
        img_files.append((img_path, cid, box))

print(f"총 샘플: {len(img_files)}")
# 예시: ('.../000123.jpg', 0, (0.51, 0.48, 0.17, 0.21))



import os
from ultralytics import YOLO
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ✅ 더미 타겟 클래스 정의
class DummyTarget:
    def __call__(self, model_output):
        return model_output.mean()

    
    
# ✅ 이미지 디렉토리에서 상위 20개 파일 가져오기
img_dir = "/home/smile/work/pbs06/test/images"
# img_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(".jpg")])[0:400]

yaml_path = "/home/smile/work/pbs06/yolov5/data/blood_cell.yaml"
class_names = yaml.safe_load(open(yaml_path))["names"]  # len=25
MAX_C = 20  # 가로 20칸


In [ ]:
from pathlib import Path
from ultralytics import YOLO
import torch, torch.nn as nn
import cv2, numpy as np, matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image

# ────────── 1. 기본 설정 ──────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_path = "/home/smile/work/pbs/model/yolov5/runsbackup_v10/250417_U2L_001_v10/blood_cell8/weights/best.pt"
class_names = {0: "RBC", 1: "WBC"}  # 클래스 ID에 따라 수정

# ────────── 2. 백본 래퍼 정의 ──────────
class YOLOv10Backbone(nn.Module):
    def __init__(self, full, stop_idx=23, target_idx=14):
        super().__init__()
        self.layers = full.model[:stop_idx]
        self.target_idx = target_idx

    def forward(self, x):
        inter = []
        cam_feat = None
        for i, m in enumerate(self.layers):
            if hasattr(m, "f") and m.f not in (None, -1):
                refs = m.f if isinstance(m.f, (list, tuple)) else [m.f]
                x = m([inter[k if k >= 0 else i + k] for k in refs])
            else:
                x = m(x)
            inter.append(x)
            if i == self.target_idx:
                cam_feat = x
        return x, cam_feat

# ────────── 3. 타겟 클래스 정의 ──────────
class YOLOBoxTarget:
    def __init__(self, box, img_size=640, top_k=8):
        self.box = box
        self.img_size = img_size
        self.top_k = top_k

    def __call__(self, activations):
        _, C, H, W = activations.shape
        x1, y1, x2, y2 = self.box
        x1 = int(x1 / self.img_size * W)
        x2 = int(x2 / self.img_size * W)
        y1 = int(y1 / self.img_size * H)
        y2 = int(y2 / self.img_size * H)
        x1, y1 = max(x1, 0), max(y1, 0)
        x2, y2 = min(x2, W), min(y2, H)

        if x1 >= x2 or y1 >= y2:
            return torch.tensor(0.0, device=activations.device)

        patch = activations[0, :, y1:y2, x1:x2]
        scores = patch.abs().mean((1, 2))
        topk = scores.topk(min(self.top_k, C))
        return patch[topk.indices].mean()

# ────────── 4. 모델 초기화 ──────────
full = YOLO(model_path).model.cpu().eval()
backbone = YOLOv10Backbone(full).to(device).eval()
target_layers = [backbone.layers[12]]
cam = GradCAMPlusPlus(model=backbone, target_layers=target_layers)
yolo_model = YOLO(model_path)

# ────────── 5. 이미지 루프 ──────────
img_dir = Path("/your/image/folder")  # 여기에 이미지 폴더 경로 설정
img_files = sorted(img_dir.glob("*.jpg"))
TOP_N = 10

for img_path in img_files[:TOP_N]:
    img = cv2.imread(str(img_path))
    img = cv2.resize(img, (640, 640))
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    tensor = torch.from_numpy(rgb.transpose(2, 0, 1)).unsqueeze(0).to(device).requires_grad_(True)

    pred = yolo_model(str(img_path), verbose=False)[0]
    if len(pred.boxes) == 0:
        continue

    # ─ 클래스별로 시각화 수행 ─
    for i in range(len(pred.boxes)):
        conf = float(pred.boxes.conf[i])
        label = int(pred.boxes.cls[i])
        if conf < 0.5:  # confidence 필터링
            continue

        box = pred.boxes.xyxy[i].cpu().tolist()
        target = [YOLOBoxTarget(box, img_size=640)]

        cam_map = cam(tensor, targets=target, eigen_smooth=False, aug_smooth=False)[0]
        cam_norm = (cam_map - cam_map.min()) / (cam_map.ptp() + 1e-8)
        cam_norm = np.power(np.clip(cam_norm, 0, 1), 1.5)
        overlay = show_cam_on_image(rgb, cam_norm, use_rgb=True, image_weight=0.5)

        # 시각화
        plt.figure(figsize=(5, 5))
        plt.imshow(overlay)
        plt.title(f"{img_path.name} | {class_names.get(label, label)} | conf={conf:.2f}")
        plt.axis("off")
        plt.show()

In [ ]:
from pytorch_grad_cam.utils.model_targets import FasterRCNNBoxScoreTarget

# ────────── 1. 전체 모델 로드 ──────────
full = YOLO("/home/smile/work/pbs/model/yolov5/runsbackup_v10/250417_U2L_001_v10/blood_cell8/weights/best.pt").model.cpu().eval()
yolo_model = YOLO("/home/smile/work/pbs/model/yolov5/runsbackup_v10/250417_U2L_001_v10/blood_cell8/weights/best.pt")  # 추론용
backbone = YOLOv10Backbone(full).eval()

# ────────── 2. CAM 대상 레이어 설정 ──────────
target_layers = [backbone.layers[12]]
cam = EigenCAM(model=backbone, target_layers=target_layers)

# ────────── 3. 이미지 루프 ──────────
for img_path, *_ in img_files:
    # 1) 이미지 전처리
    img = cv2.imread(img_path)
    img = cv2.resize(img, (640, 640))
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    tensor = torch.from_numpy(rgb.transpose(2, 0, 1)).unsqueeze(0)

    # 2) YOLO 모델로 예측
    pred = yolo_model(img_path, verbose=False)[0]
    if len(pred.boxes) == 0:
        continue  # 예측된 객체 없으면 건너뜀
    
    targets = []
    for box_tensor, cls_tensor in zip(pred.boxes.xyxy, pred.boxes.cls):
        box = box_tensor.cpu().tolist()
        cls = int(cls_tensor)
        targets.append(FasterRCNNBoxScoreTarget(box, cls))

        # 4) CAM 계산
        cam_map = cam(tensor, targets=target)[0]
        overlay = show_cam_on_image(rgb, cam_map, use_rgb=True, image_weight=0.25)

        # 5) 시각화
        plt.imshow(overlay)
        plt.title(f"{Path(img_path).name} | label={label}")
        plt.axis("off")
        plt.show()

#     # 3) 가장 높은 confidence box를 CAM 타겟으로 설정
#     box = pred.boxes.xyxy[0].cpu().tolist()  # [x1, y1, x2, y2] 리스트
#     label = int(pred.boxes.cls[0])
#     target = [FasterRCNNBoxScoreTarget(box, label)]  

#     # 4) CAM 계산
#     cam_map = cam(tensor, targets=target)[0]
#     overlay = show_cam_on_image(rgb, cam_map, use_rgb=True, image_weight=0.25)

#     # 5) 시각화
#     plt.imshow(overlay)
#     plt.title(f"{Path(img_path).name} | label={label}")
#     plt.axis("off")
#     plt.show()


In [ ]:
from pathlib import Path
from ultralytics import YOLO
import torch, torch.nn as nn, cv2, numpy as np, matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image

# ────────── 1. YOLOv10 Detect 전 백본 래퍼 ──────────
class YOLOv10Backbone(nn.Module):
    def __init__(self, full, stop_idx=23):
        super().__init__()
        self.layers = full.model[:stop_idx]

    def forward(self, x):
        inter = []
        for i, m in enumerate(self.layers):
            if hasattr(m, "f") and m.f not in (None, -1):
                refs = m.f if isinstance(m.f, (list, tuple)) else [m.f]
                x = m([inter[k if k >= 0 else i + k] for k in refs])
            else:
                x = m(x)
            inter.append(x)
        return x

# ────────── 2. 모델 및 백본 초기화 ──────────
full = YOLO("/home/smile/work/pbs/model/yolov5/runsbackup_v10/250417_U2L_001_v10/blood_cell8/weights/best.pt").model.cpu().eval()
backbone = YOLOv10Backbone(full).eval()
target_layers = [backbone.layers[12]]  # neck 중간

# ────────── 3. GradCAM++ 객체 생성 ──────────
cam = GradCAMPlusPlus(model=backbone, target_layers=target_layers)

# ────────── 4. 다중 채널 Target 클래스 ──────────
class MultiChannelTarget:
    def __init__(self, ch_list): self.ch_list = ch_list
    def __call__(self, feat):
        return torch.stack([feat[0, ch].mean() for ch in self.ch_list]).mean()

# ────────── 5. 이미지 루프 (CAM++ 시각화만) ──────────
TOP_K = 5  # 사용할 채널 수

for img_path, *_ in img_files[:6]:  # 앞에서 6장만 예시
    img = cv2.imread(img_path)
    img = cv2.resize(img, (640, 640))
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    tensor = torch.from_numpy(rgb.transpose(2, 0, 1)).unsqueeze(0).requires_grad_(True)

    with torch.no_grad():
        _ = backbone(tensor)
        feat = cam.activations  # CAM 대상 레이어 출력
        scores = feat[0].abs().mean((1, 2))
        topk = scores.topk(min(TOP_K, scores.shape[0]))
        ch_ids = topk.indices.tolist()

    target = MultiChannelTarget(ch_ids)
    cam_map = cam(tensor, targets=[target], eigen_smooth=False, aug_smooth=False)[0]
    cam_norm = (cam_map - cam_map.min()) / (cam_map.ptp() + 1e-8)
    overlay = show_cam_on_image(rgb, cam_norm, use_rgb=True, image_weight=0.5)

    plt.figure(figsize=(5, 5))
    plt.imshow(overlay)
    plt.axis("off")
    plt.title(f"{Path(img_path).name} | chs {ch_ids} | CAM++")
    plt.show()


In [ ]:
from pathlib import Path
from ultralytics import YOLO
import torch, torch.nn as nn, cv2, numpy as np, matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image

# ────────── 1. YOLOv10 Detect 전 백본 래퍼 ──────────
class YOLOv10Backbone(nn.Module):
    def __init__(self, full, stop_idx=23):
        super().__init__()
        self.layers = full.model[:stop_idx]

    def forward(self, x):
        inter = []
        for i, m in enumerate(self.layers):
            if hasattr(m, "f") and m.f not in (None, -1):
                refs = m.f if isinstance(m.f, (list, tuple)) else [m.f]
                x = m([inter[k if k >= 0 else i + k] for k in refs])
            else:
                x = m(x)
            inter.append(x)
        return x

# ────────── 2. 모델 및 백본 초기화 ──────────
full = YOLO("/home/smile/work/pbs/model/yolov5/runsbackup_v10/250417_U2L_001_v10/blood_cell8/weights/best.pt").model.cpu().eval()
backbone = YOLOv10Backbone(full).eval()
target_layers = [backbone.layers[12]]  # neck 중간

# ────────── 3. GradCAM++ 객체 생성 ──────────
cam = GradCAMPlusPlus(model=backbone, target_layers=target_layers)

# ────────── 4. 단일 채널 Target 클래스 ──────────
class ChannelMeanTarget:
    def __init__(self, ch): self.ch = ch
    def __call__(self, feat):
        idx = min(self.ch, feat.shape[1] - 1)
        return feat[0, idx].mean()

# ────────── 5. 이미지 루프 (CAM++ 시각화만) ──────────
for img_path, *_ in img_files[:200]:  # 앞에서 6장만 예시
    img = cv2.imread(img_path)
    img = cv2.resize(img, (640, 640))
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    tensor = torch.from_numpy(rgb.transpose(2, 0, 1)).unsqueeze(0).requires_grad_(True)

    with torch.no_grad():
        feat = backbone(tensor)
        ch_id = feat[0].abs().mean((1, 2)).argmax().item()

    target = ChannelMeanTarget(ch_id)
    cam_map = cam(tensor, targets=[target], eigen_smooth=False, aug_smooth=False)[0]
    cam_norm = (cam_map - cam_map.min()) / (cam_map.ptp() + 1e-8)
    overlay = show_cam_on_image(rgb, cam_norm, use_rgb=True, image_weight=0.5)

    plt.figure(figsize=(5, 5))
    plt.imshow(overlay)
    plt.axis("off")
    plt.title(f"{Path(img_path).name} | ch {ch_id} | CAM++")
    plt.show()

In [ ]:
# ────────── 라이브러리 ──────────
from pathlib import Path
from ultralytics import YOLO
import torch, torch.nn as nn, cv2, numpy as np, matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, ScoreCAM, EigenCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

# ────────── 1. Detect 전까지 백본 래퍼 ──────────
class YOLOv10Backbone(nn.Module):
    def __init__(self, full, stop_idx=23):
        super().__init__();  self.layers = full.model[:stop_idx]
    def forward(self, x):
        inter = []
        for i,m in enumerate(self.layers):
            if hasattr(m,"f") and m.f not in (None,-1):
                refs = m.f if isinstance(m.f,(list,tuple)) else [m.f]
                x = m([inter[k if k>=0 else i+k] for k in refs])
            else: x = m(x)
            inter.append(x)
        return x                                  # [1,C,H,W]

# ────────── 2. 모델 & 백본 준비 ──────────
full     = YOLO("/home/smile/work/pbs/model/yolov5/runsbackup_v10/250417_U2L_001_v10/blood_cell8/weights/best.pt").model.cpu().eval()
backbone = YOLOv10Backbone(full).eval()

target_layers = [backbone.layers[12]]            # neck 중간 (256ch)

# ────────── 3. CAM 객체 목록 만들기 ──────────
cam_dict = {
    "Grad-CAM"      : GradCAM(model=backbone,       target_layers=target_layers),
    "Grad-CAM++"    : GradCAMPlusPlus(model=backbone,target_layers=target_layers),
    "Score-CAM"     : ScoreCAM(model=backbone,      target_layers=target_layers),
    "Eigen-CAM"     : EigenCAM(model=backbone,      target_layers=target_layers)
}

# ────────── 4. 단일-채널 Target 클래스 ──────────
class ChannelMeanTarget:
    def __init__(self, ch): self.ch = ch
    def __call__(self, feat):          # 안전 보정
        idx = min(self.ch, feat.shape[1]-1)
        return feat[0, idx].mean()

# ────────── 5. 한 장만 시험 (여러 장이면 for-loop) ──────────
img_path = img_files[0][0]             # 첫 번째 파일 예시

# 5-1) 전처리 ------------------------------------------------------------
img  = cv2.imread(img_path)
img  = cv2.resize(img,(640,640))
rgb  = cv2.cvtColor(img,cv2.COLOR_BGR2RGB).astype(np.float32)/255.
tensor = torch.from_numpy(rgb.transpose(2,0,1)).unsqueeze(0).requires_grad_(True)

# 5-2) 가장 강한 채널 하나 선택 -----------------------------------------
with torch.no_grad():
    feat = backbone(tensor)            # [1,C,H,W]
    ch_id = feat[0].abs().mean((1,2)).argmax().item()   # top-1
target = ChannelMeanTarget(ch_id)

# 5-3) 각 CAM 계산 & 시각화용 리스트 --------------------------------------
overlays = []
titles   = []
for name, cam_obj in cam_dict.items():
    cam_map = cam_obj(tensor, targets=[target],
                      eigen_smooth=False, aug_smooth=False)[0]
    cam_norm = (cam_map-cam_map.min())/(cam_map.ptp()+1e-8)
    overlay  = show_cam_on_image(rgb, cam_norm, use_rgb=True)
    overlays.append(overlay);  titles.append(name)

# ────────── 6. 가로로 플롯 ----------------------------------------------
plt.figure(figsize=(14,4))
for i,(ov,ttl) in enumerate(zip(overlays,titles)):
    plt.subplot(1,len(overlays),i+1)
    plt.imshow(ov); plt.axis('off'); plt.title(ttl, fontsize=12)
plt.suptitle(f"{Path(img_path).name}  |  layer 12  |  ch {ch_id}", fontsize=14)
plt.show()

In [ ]:
print(full.model[12])

In [ ]:
top_idx

In [ ]:
top_idx

In [ ]:
pred[:, 4]

In [ ]:
pred[0]

In [ ]:
import torch.nn as nn

class YOLOv10Backbone(nn.Module):
    def __init__(self, full, stop_idx=23):
        super().__init__()
        self.layers = full.model[:stop_idx]     # Detect 직전까지 잘라서 보관
    def forward(self, x):
        inter = []
        for i, layer in enumerate(self.layers):
            if hasattr(layer, "f") and layer.f not in (None, -1):
                f_idx = layer.f if isinstance(layer.f, (list, tuple)) else [layer.f]
                x_in = [inter[k if k >= 0 else i + k] for k in f_idx]
                x = layer(x_in)
            else:
                x = layer(x)
            inter.append(x)
        return x        


from ultralytics import YOLO
import torch, cv2, numpy as np, matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image

# 1) 전체 YOLOv10 불러오기
full = YOLO("/home/smile/work/pbs/model/yolov5/runsbackup_v10/250417_U2L_001_v10/blood_cell8/weights/best.pt")
full = full.model.cpu().eval()

# 2) Detect 이전까지만 수행하는 래퍼 모델
backbone = YOLOv10Backbone(full, stop_idx=23).eval()

target_layers = [backbone.layers[12]]          # neck 중간
cam = GradCAMPlusPlus(model=backbone, target_layers=target_layers)


for img_path, *_ in img_files:
    # 전처리 ------------------------------------------------------------
    img  = cv2.imread(img_path)
    img  = cv2.resize(img,(640,640))
    rgb  = cv2.cvtColor(img,cv2.COLOR_BGR2RGB).astype(np.float32)/255.0
    tensor = torch.from_numpy(rgb.transpose(2,0,1)).unsqueeze(0).requires_grad_(True)

    # 타깃 채널 계산 ----------------------------------------------------
    with torch.no_grad():
        C = backbone(tensor).shape[1]      # 레이어 21의 채널 개수
    ch_id = 128 if 128 < C else C//2       # 자동 보정

    class ChannelMeanTarget:
        def __init__(self,ch): self.ch=ch
        def __call__(self,f):
            idx = min(self.ch, f.shape[1]-1)
            return f[0, idx].mean()

    target = ChannelMeanTarget(ch_id)
    # target = DummyTarget()              # ← 언제든 안전 대체 가능

    cam_map = cam(tensor, targets=[target])[0]
    overlay = show_cam_on_image(rgb, cam_map, use_rgb=True)

    plt.imshow(overlay); plt.axis("off")
    plt.title(f"{Path(img_path).name} | ch={ch_id}")
    plt.show()


In [ ]:
# … (앞부분 동일) …

TOP_K = 120                      # ← 상위 8개 채널 평균
for img_path, *_ in img_files:
    # ── 1) 전처리 ──────────────────────────────────────────
    img  = cv2.imread(img_path)
    img  = cv2.resize(img, (640, 640))
    rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.
    tensor = torch.from_numpy(rgb.transpose(2,0,1)).unsqueeze(0).requires_grad_(True)

    # ── 2) 상위 K채널 뽑기 ─────────────────────────────────
    with torch.no_grad():
        feat = backbone(tensor)                # [1,C,H,W]
        C    = feat.shape[1]
        scores  = feat[0].abs().mean((1,2))    # 채널별 평균 활성
        top_ids = scores.topk(min(TOP_K,C)).indices.tolist()

    print(f"{Path(img_path).name} | C={C} | top{TOP_K}={top_ids}")

    # ➊ 8개 채널 CAM 계산 & 누적합 -----------------------------------
    fused = None
    for ch in top_ids:
        cam_map = cam(tensor,
                      targets=[ChannelMeanTarget(ch)],
                      eigen_smooth=False, aug_smooth=False)[0]   # 옵션 OFF로 속도↑
        fused = cam_map if fused is None else fused + cam_map

    fused_cam = fused / len(top_ids)          # 평균
    # (최대값으로 보려면 fused_cam = np.maximum.reduce(cam_list))

    # ➋ 시각화 --------------------------------------------------------
    overlay = show_cam_on_image(rgb, fused_cam, use_rgb=True)
    plt.imshow(overlay); plt.axis("off")
    plt.title(f"{Path(img_path).name} | fused top-{TOP_K}")
    plt.show()


In [ ]:
# … (앞부분 동일) …

TOP_K = 1                      # ← 상위 8개 채널 평균
for img_path, *_ in img_files:
    # ── 1) 전처리 ──────────────────────────────────────────
    img  = cv2.imread(img_path)
    img  = cv2.resize(img, (640, 640))
    rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.
    tensor = torch.from_numpy(rgb.transpose(2,0,1)).unsqueeze(0).requires_grad_(True)

    # ── 2) 상위 K채널 뽑기 ─────────────────────────────────
    with torch.no_grad():
        feat = backbone(tensor)                # [1,C,H,W]
        C    = feat.shape[1]
        scores  = feat[0].abs().mean((1,2))    # 채널별 평균 활성
        top_ids = scores.topk(min(TOP_K,C)).indices.tolist()

    print(f"{Path(img_path).name} | C={C} | top{TOP_K}={top_ids}")

    # ➊ 8개 채널 CAM 계산 & 누적합 -----------------------------------
    fused = None
    for ch in top_ids:
        cam_map = cam(tensor,
                      targets=[ChannelMeanTarget(ch)],
                      eigen_smooth=False, aug_smooth=False)[0]   # 옵션 OFF로 속도↑
        fused = cam_map if fused is None else fused + cam_map

    fused_cam = fused / len(top_ids)          # 평균
    # (최대값으로 보려면 fused_cam = np.maximum.reduce(cam_list))

    # ➋ 시각화 --------------------------------------------------------
    overlay = show_cam_on_image(rgb, fused_cam, use_rgb=True)
    plt.imshow(overlay); plt.axis("off")
    plt.title(f"{Path(img_path).name} | fused top-{TOP_K}")
    plt.show()


In [ ]:
# ───────────────── 라이브러리 & 백본 래퍼 (생략: 이전과 동일) ─────────────

# 안전한 단일-채널 Target  ────────────────────────────────────────────────
class ChannelMeanTarget:
    """
    - 배치 차원 개수(N)가 바뀌어도 안전
    - self.ch 가 채널 수보다 크면 마지막 채널로 자동 보정
    """
    def __init__(self, ch: int): self.ch = ch

    def __call__(self, feat):
        # feat shape 예: [B,C,H,W]  또는  [C,H,W]
        if feat.ndim == 4:
            B, C, *_ = feat.shape
            idx = min(self.ch, C-1)
            return feat[:, idx].mean()          # B개 모두 평균
        elif feat.ndim == 3:
            C, *_ = feat.shape
            idx = min(self.ch, C-1)
            return feat[idx].mean()
        else:
            raise RuntimeError(f"Unexpected feat shape: {feat.shape}")

# ────────── 모델 & CAM 준비 (full, backbone, target_layers … 동일) ──────
cam = GradCAMPlusPlus(model=backbone,
                      target_layers=target_layers)           # GPU면 True, 버전별 옵션

# ────────── 이미지 루프  ────────────────────────────────────────────────
for img_path, *_ in img_files[:5]:
    # 1) 전처리
    img  = cv2.imread(img_path)
    img  = cv2.resize(img, (640,640))
    rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)/255.
    t    = torch.from_numpy(rgb.transpose(2,0,1)).unsqueeze(0).requires_grad_(True)

    # 2) 레이어 채널 수 파악
    with torch.no_grad():
        C = backbone(t).shape[1]

    # 3) 모든 채널 CAM 평균 (메모리 절약형 누적 합)
    fused = None
    for ch in range(C):
        cam_map = cam(t, targets=[ChannelMeanTarget(ch)],
                      eigen_smooth=False, aug_smooth=False)[0]   # 옵션 OFF로 속도 ↑
        fused = cam_map if fused is None else fused + cam_map

    fused_cam = fused / C            # 전체 평균
    overlay   = show_cam_on_image(rgb, fused_cam, use_rgb=True)

    # 4) 시각화
    plt.imshow(overlay); plt.axis("off")
    plt.title(f"{Path(img_path).name} | fused {C}ch avg")
    plt.show()


In [ ]:
!pip install --upgrade pytorch-grad-cam  

In [ ]:
with torch.no_grad():
    out = backbone(t)             # t = 임의 입력
print("feature shape :", out.shape)        # ex) [1, 20, H, W]
print("channel id 선택 :", min(128, out.shape[1]-1))

In [ ]:
# 0. 라이브러리 -----------------------------------------------------------
from pathlib import Path
from ultralytics import YOLO
import torch, torch.nn as nn
import cv2, numpy as np, matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image

# 1. YOLOv10 백본 ---------------------------------------------------------
class YOLOv10Backbone(nn.Module):
    def __init__(self, full, stop_idx=23):
        super().__init__(); self.layers = full.model[:stop_idx]
    def forward(self, x):
        inter = []
        for i, m in enumerate(self.layers):
            if hasattr(m, "f") and m.f not in (None, -1):
                f_idx = m.f if isinstance(m.f, (list, tuple)) else [m.f]
                x = m([inter[k if k >= 0 else i+k] for k in f_idx])
            else:
                x = m(x)
            inter.append(x)
        return x                                  # [1,C,H,W]

# 2. Target 클래스 --------------------------------------------------------
class ChannelMeanTarget:   # 단일 채널 평균
    def __init__(self, ch): self.ch = ch
    def __call__(self, f):  return f[0, self.ch].mean()

class TopKChannelsTarget:   # 상위 K개 채널 평균
    def __init__(self, k=8): self.k = k
    def __call__(self, f):
        idx = f[0].abs().mean((1,2)).topk(self.k).indices
        return f[0, idx].mean()

class BoxCenterTarget:      # 박스 중심
    def __init__(self, xc, yc, h, w):
        self.u, self.v = int(xc*w), int(yc*h)
    def __call__(self, f):  return f[0,:,self.v,self.u].mean()

class DetectScoreTarget:    # Detect raw 점수
    def __init__(self, raw, a=0, cls=0, y=0, x=0):
        self.score = raw[a,y,x,5+cls]
    def __call__(self, _):  return self.score

# 3. 모델 & CAM ----------------------------------------------------------
full = YOLO("/home/smile/work/pbs/model/yolov5/runsbackup_v10/250417_U2L_001_v10/blood_cell8/weights/best.pt").model.cpu().eval()
backbone = YOLOv10Backbone(full).eval()
target_layers = [backbone.layers[21]]
cam = GradCAMPlusPlus(backbone, target_layers)

# 4. 이미지 루프 ---------------------------------------------------------
for img_path, cls_id, (xc, yc, _, _) in img_files[:10]:
    img = cv2.imread(img_path)
    img = cv2.resize(img, (640,640))
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)/255.0
    t   = torch.from_numpy(rgb.transpose(2,0,1)).unsqueeze(0).requires_grad_(True)

    # --- Target 선택 ---
    C = backbone(t).shape[1]                # 현재 레이어 채널 수
    ch_id = min(128, C-1)                   # 범위 안전
    target = ChannelMeanTarget(ch_id)       # ① 단일 채널
    # target = TopKChannelsTarget(k=8)      # ②
    # h,w = backbone(t).shape[-2:]; target = BoxCenterTarget(xc,yc,h,w)  # ③
    # feat = backbone(t); raw = full.model.model[23](feat)[0]
    # target = DetectScoreTarget(raw, a=0, cls=cls_id)                  # ④
    # --------------------

    cam_map = cam(t, targets=[target],
                  eigen_smooth=True, aug_smooth=True)[0]
    overlay = show_cam_on_image(rgb, cam_map, use_rgb=True, alpha=0.6)

    plt.imshow(overlay); plt.axis("off")
    plt.title(f"{Path(img_path).name} | {target.__class__.__name__}")
    plt.show()


In [ ]:
from ultralytics import YOLO
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image

# ✅ 모델 로딩 (CPU 기준)
yolo = YOLO("/home/smile/work/pbs/model/yolov5/runsbackup_v10/250417_U2L_001_v10/blood_cell8/weights/best.pt")
model = yolo.model.cpu().eval()

# ✅ 타겟 레이어 (예: neck 중간 C2f)
target_layers = [model.model[13]]

# ✅ GradCAM++ 인스턴스
cam = GradCAMPlusPlus(model=model, target_layers=target_layers)

# ✅ Dummy Target 정의
class DummyTarget:
    def __call__(self, model_output):
        return model_output.sum()

# ✅ 이미지 반복
for img_name in img_files:
    img_path = img_name[0]
    print(f"📌 Processing: {img_path}")

    # 이미지 전처리
    img = cv2.imread(img_path)
    img = cv2.resize(img, (640, 640))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    input_image = img_rgb.astype(np.float32) / 255.0
    img_tensor = torch.from_numpy(input_image.transpose(2, 0, 1)).unsqueeze(0).cpu().requires_grad_(True)

    # ✅ 수동 forward (Detect 전까지 수행)
    intermediates = []
    x = img_tensor
    for i, layer in enumerate(model.model):
        # (1) f 가 리스트/튜플/양수일 때만 skip-inputs 수집
        if hasattr(layer, "f") and layer.f not in (None, -1):
            f_idx = layer.f if isinstance(layer.f, (list, tuple)) else [layer.f]
            inputs = []
            for j in f_idx:
                idx = j if j >= 0 else i + j
                if 0 <= idx < len(intermediates):
                    inputs.append(intermediates[idx])
                else:
                    print(f"⚠️  invalid idx {idx} at layer {i}")
            if not inputs:
                x = None; break
            x = layer(inputs)
        else:
            # (2) 일반 레이어 또는 f == -1
            x = layer(x)

        intermediates.append(x)
        if i == 22:          # Detect 직전까지만
            break

    if x is None:
        continue  # 잘못된 이미지 건너뜀

    # ✅ GradCAM++ 실행
    with torch.set_grad_enabled(True):
        grayscale_cam = cam(input_tensor=img_tensor, targets=[DummyTarget()])[0]

    # ✅ CAM 시각화
    cam_overlay = show_cam_on_image(input_image, grayscale_cam, use_rgb=True)
    plt.imshow(cam_overlay)
    plt.axis("off")
    plt.title(f"CAM: {img_path.split('/')[-1]}")
    plt.show()


In [ ]:
for idx, m in enumerate(model.model[:5]):  # 앞 5개 레이어 예시
    print(idx, type(m).__name__, getattr(m, "f", None))

In [ ]:
from ultralytics import YOLO
import torch

model = YOLO("yolov10s.pt").model.cuda().eval()

img = torch.randn(1, 3, 640, 640).cuda().requires_grad_(True)

# 이건 추론용 후처리 포함된 것
output1 = model(img)  # output1.requires_grad → False

# Detect 모듈 이전까지만 forward
output2 = model.model[:23](img)  # 이건 gradient 추적됨
print(output2.requires_grad)  # True

In [ ]:
from matplotlib import pyplot as plt

# class 별로 묶기
from collections import defaultdict
class_img_map = defaultdict(list)

for img_path, cid, box in img_files:
    class_img_map[cid].append((img_path, box))

# 시각화
for cid in sorted(class_img_map.keys()):
    print(f"\n🧩 Class {cid}: {class_names[cid]}")
    samples = class_img_map[cid][:20]

    fig, axs = plt.subplots(3, len(samples), figsize=(len(samples)*1.5, 5))
    fig.suptitle(f"Class {cid}: {class_names[cid]}", fontsize=14)

    for idx, (img_path, box) in enumerate(samples):
        # 이미지 로딩 및 전처리
        img = cv2.imread(img_path)
        img = cv2.resize(img, (384, 384))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        input_image = img_rgb.astype(np.float32) / 255.0
        img_tensor = torch.from_numpy(input_image.transpose(2, 0, 1)).unsqueeze(0).cuda()
        img_tensor.requires_grad_()

        # Grad-CAM
        with torch.set_grad_enabled(True):
            grayscale_cam = cam(input_tensor=img_tensor, targets=[DummyTarget()])[0]
            cam_overlay = show_cam_on_image(input_image, grayscale_cam, use_rgb=True)

        # Crop
        xc, yc, bw, bh = box
        x1 = int((xc - bw/2) * 384);  y1 = int((yc - bh/2) * 384)
        x2 = int((xc + bw/2) * 384);  y2 = int((yc + bh/2) * 384)
        x1, y1 = max(x1, 0), max(y1, 0)
        x2, y2 = min(x2, 383), min(y2, 383)

        crop_rgb = img_rgb[y1:y2, x1:x2]
        crop_overlay = cam_overlay[y1:y2, x1:x2]
        crop_cam = grayscale_cam[y1:y2, x1:x2]

        axs[0, idx].imshow(crop_rgb)
        axs[1, idx].imshow(crop_overlay)
        axs[2, idx].imshow(crop_cam, cmap='jet')

        for row in range(3):
            axs[row, idx].axis('off')
            if row == 2:
                axs[row, idx].set_title(f"{idx+1}", fontsize=8)

    plt.tight_layout()
    plt.show()

In [ ]:
out = model.model(img_tensor)   # GradCAM에서 받는 원시 출력
print(type(out), len(out) if isinstance(out, (list,tuple)) else out.shape)

In [ ]:
import os
from ultralytics import YOLO
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ✅ 더미 타겟 클래스 정의
class DummyTarget:
    def __call__(self, model_output):
        return model_output.mean()

# ✅ 이미지 디렉토리에서 상위 20개 파일 가져오기
img_dir = "/home/smile/work/pbs06/test/images"
img_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(".jpg")])[0:400]

# ✅ YOLO 모델 불러오기
model = YOLO("/home/smile/work/pbs/model/yolov5/runsbackup_v10/250403_U2L_001_v10/blood_cell2/weights/best.pt")
# model = YOLO("/home/smile/work/pbs/model/yolov5/runsbackup_v10/250402_U2L_001_v10/blood_cell/weights/best.pt")



model = model.cuda().eval()

# ✅ GradCAM++ 설정
target_layers = [model.model.model[13]]
cam = GradCAMPlusPlus(model=model.model, target_layers=target_layers)

# ✅ 이미지별 반복
for img_name in img_files:
    img_path = os.path.join(img_dir, img_name)
    print(f"📌 Processing: {img_path}")

    # 이미지 로딩 및 전처리
    img = cv2.imread(img_path)
    img = cv2.resize(img, (384, 384))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    input_image = img_rgb.astype(np.float32) / 255.0
    img_tensor = torch.from_numpy(input_image.transpose(2, 0, 1)).unsqueeze(0).cuda()
    img_tensor.requires_grad_()

    # CAM 계산
    with torch.set_grad_enabled(True):
        grayscale_cam = cam(input_tensor=img_tensor, targets=[DummyTarget()])[0]
        cam_overlay = show_cam_on_image(input_image, grayscale_cam, use_rgb=True)

    # ✅ 원본, Overlay, Raw CAM 시각화
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.imshow(img_rgb)
    plt.title("Original Image")
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(cam_overlay)
    plt.title("CAM Overlay")
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(grayscale_cam, cmap='jet')
    plt.title("Raw CAM")
    plt.axis('off')

    plt.suptitle(f"Image: {img_name}")
    plt.tight_layout()
    plt.show()

    # 박스 예측
    results = model(img_tensor)[0]
    boxes = results.boxes

    # 박스별 CAM 시각화
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        cls = int(box.cls.item())
        cam_crop = grayscale_cam[y1:y2, x1:x2]
        overlay_crop = cam_overlay[y1:y2, x1:x2]
        crop_rgb = img_rgb[y1:y2, x1:x2]

        plt.figure(figsize=(9, 3))
        plt.suptitle(f"{img_name} - Box {i} (Class {cls})")
        plt.subplot(1, 3, 1)
        plt.imshow(crop_rgb)
        plt.title("Original Crop")
        plt.axis('off')

        plt.subplot(1, 3, 2)
        plt.imshow(overlay_crop)
        plt.title("Overlay Crop")
        plt.axis('off')

        plt.subplot(1, 3, 3)
        plt.imshow(cam_crop, cmap='jet')
        plt.title("Raw CAM Crop")
        plt.axis('off')

        plt.tight_layout()
        plt.show()


In [ ]:
import os
from ultralytics import YOLO
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ✅ 더미 타겟 클래스 정의
class DummyTarget:
    def __call__(self, model_output):
        return model_output.mean()

# ✅ 이미지 디렉토리에서 상위 20개 파일 가져오기
img_dir = "/home/smile/work/pbs06/test/images"
img_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(".jpg")])[:20]

# ✅ YOLO 모델 불러오기
model = YOLO("/home/smile/work/pbs/model/yolov5/runsbackup_v10/250403_U2L_001_v10/blood_cell2/weights/best.pt")
model = model.cuda().eval()

# ✅ GradCAM++ 설정
target_layers = [model.model.model[13]]
cam = GradCAMPlusPlus(model=model.model, target_layers=target_layers)

# ✅ 이미지별 반복
for img_name in img_files:
    img_path = os.path.join(img_dir, img_name)
    print(f"📌 Processing: {img_path}")

    # 이미지 로딩 및 전처리
    img = cv2.imread(img_path)
    img = cv2.resize(img, (384, 384))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    input_image = img_rgb.astype(np.float32) / 255.0
    img_tensor = torch.from_numpy(input_image.transpose(2, 0, 1)).unsqueeze(0).cuda()
    img_tensor.requires_grad_()

    # CAM 계산
    with torch.set_grad_enabled(True):
        grayscale_cam = cam(input_tensor=img_tensor, targets=[DummyTarget()])[0]
        cam_overlay = show_cam_on_image(input_image, grayscale_cam, use_rgb=True)

    # 전체 이미지 CAM 출력
    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(cam_overlay)
    plt.title("Full Image CAM Overlay")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(grayscale_cam, cmap='jet')
    plt.title("Full Image Raw CAM")
    plt.axis('off')
    plt.suptitle(f"Image: {img_name}")
    plt.tight_layout()
    plt.show()

    # 박스 예측
    results = model(img_tensor)[0]
    boxes = results.boxes

    # 박스별 CAM 시각화
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        cls = int(box.cls.item())
        cam_crop = grayscale_cam[y1:y2, x1:x2]
        overlay_crop = cam_overlay[y1:y2, x1:x2]

        plt.figure(figsize=(6, 3))
        plt.suptitle(f"{img_name} - Box {i} (Class {cls})")
        plt.subplot(1, 2, 1)
        plt.imshow(overlay_crop)
        plt.title("Overlay")
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(cam_crop, cmap='jet')
        plt.title("Raw CAM")
        plt.axis('off')

        plt.tight_layout()
        plt.show()


In [ ]:
import os
import time
from ultralytics import YOLO
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
import torch
import cv2
import numpy as np

# ✅ 더미 타겟 클래스 정의
class DummyTarget:
    def __call__(self, model_output):
        return model_output.mean()

# ✅ 테스트 이미지 디렉토리 (최대 300개까지)
img_dir = "/home/smile/work/pbs06/test/images"
img_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(".jpg")])[:300]

# ✅ YOLO 모델 로딩
model = YOLO("/home/smile/work/pbs/model/yolov5/runsbackup_v10/250403_U2L_001_v10/blood_cell2/weights/best.pt")
model = model.cuda().eval()

# ✅ GradCAM++ 설정
target_layers = [model.model.model[13]]  # 보통 마지막 Conv 레이어
cam = GradCAMPlusPlus(model=model.model, target_layers=target_layers)

# ✅ 전체 루프 시간 측정
start_time = time.time()

for img_name in img_files:
    img_path = os.path.join(img_dir, img_name)

    # 이미지 로딩 및 전처리
    img = cv2.imread(img_path)
    img = cv2.resize(img, (384, 384))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    input_image = img_rgb.astype(np.float32) / 255.0
    img_tensor = torch.from_numpy(input_image.transpose(2, 0, 1)).unsqueeze(0).cuda()
    img_tensor.requires_grad_()

    # Grad-CAM 계산
    with torch.set_grad_enabled(True):
        grayscale_cam = cam(input_tensor=img_tensor, targets=[DummyTarget()])[0]
        # cam_overlay = show_cam_on_image(input_image, grayscale_cam, use_rgb=True)

#     # 박스 예측
#     results = model(img_tensor)[0]
#     boxes = results.boxes

#     # 박스별 CAM 자르기 (시각화 없이 연산만)
#     for i, box in enumerate(boxes):
#         x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
#         _ = grayscale_cam[y1:y2, x1:x2]
#         _ = cam_overlay[y1:y2, x1:x2]

end_time = time.time()
elapsed = end_time - start_time

print(f"\n✅ GradCAM++ 전체 처리 시간 (300개 이미지): {elapsed:.2f}초")
print(f"⏱️ 평균 이미지당 처리 시간: {elapsed / len(img_files):.4f}초")

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import FasterRCNNBoxScoreTarget
import torch
import cv2
import numpy as np
from pytorch_grad_cam.utils.model_targets import FasterRCNNBoxScoreTarget

img_path = '/home/smile/work/pbs/api/1812741.jpg'
img = cv2.imread(img_path)
img = cv2.resize(img, (384, 384))
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img_tensor = torch.from_numpy(img_rgb.transpose(2, 0, 1)).float().div(255.0).unsqueeze(0).cuda()


# 모델 로드
model = YOLO('/home/smile/work/pbs/model/yolov5/runsbackup_v10/250403_U2L_001_v10/blood_cell2/weights/best.pt')
model = model.cuda()
model.eval()

# 내부 모델 구조 접근
internal_model = model.model  # 실제 PyTorch Module

# Grad-CAM 타겟 레이어 설정
target_layers = [internal_model.model[-2]]  # Neck의 마지막 Conv layer (예시)

# CAM 인스턴스 생성
cam = GradCAM(model=internal_model, target_layers=target_layers)

# 예측 수행
results = model(img_tensor)[0]  # List[DetectionOutput]

# 가장 높은 confidence의 첫 번째 box 선택
boxes = results.boxes
top_idx = boxes.conf.argmax()
target_class = int(boxes.cls[top_idx].item())

# Grad-CAM 수행
grayscale_cam = cam(input_tensor=img_tensor, targets=[FasterRCNNBoxScoreTarget(target_class=target_class)])[0]
visualization = show_cam_on_image(img_rgb.astype(np.float32) / 255., grayscale_cam, use_rgb=True)

# 시각화 출력
plt.imshow(visualization)
plt.axis('off')
plt.show()

In [ ]:
from ultralytics import YOLO
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam import GradCAMPlusPlus, EigenCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ✅ 1. YOLO에 맞춘 커스텀 Grad-CAM Target 정의
class YoloBoxScoreTarget:
    def __init__(self, class_idx):
        self.class_idx = class_idx

    def __call__(self, model_output):
        # model_output: (B, A, H, W, 5 + num_classes)
        class_scores = model_output[..., 5 + self.class_idx]
        return class_scores.mean()

# ✅ 2. 이미지 불러오기
img_path = '/home/smile/work/pbs/api/1812741.jpg'
img = cv2.imread(img_path)
img = cv2.resize(img, (384, 384))
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
input_image = img_rgb.astype(np.float32) / 255.0
img_tensor = torch.from_numpy(input_image.transpose(2, 0, 1)).unsqueeze(0).cuda()
img_tensor.requires_grad_()

# ✅ 3. YOLO 모델 불러오기
model = YOLO('/home/smile/work/pbs/model/yolov5/runsbackup_v10/250403_U2L_001_v10/blood_cell2/weights/best.pt')
model = model.cuda()
model.eval()

# ✅ 4. Grad-CAM 대상 레이어 설정 (Detect 이전 마지막 conv layer)
internal_model = model.model  # 실제 nn.Module
target_layers = [internal_model.model[14]]  # 예: backbone 초반 Conv

# ✅ 5. Grad-CAM 인스턴스 생성
# cam = GradCAM(model=internal_model, target_layers=target_layers)
cam = GradCAMPlusPlus(model=internal_model, target_layers=target_layers)
# cam = EigenCAM(model=internal_model, target_layers=target_layers)

# ✅ 6. Forward + Grad-CAM 실행
target_class = 0  # ← RBC 클래스
targets = [YoloBoxScoreTarget(class_idx=target_class)]

with torch.enable_grad():
    grayscale_cam = cam(input_tensor=img_tensor, targets=targets)[0]  # CAM 생성

# ✅ 7. 시각화 및 출력 (CAM 원본과 입힌 이미지 모두 표시)
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

# CAM heatmap만 (Grayscale CAM)
ax[0].imshow(grayscale_cam, cmap='jet')
ax[0].set_title("Raw CAM (activation)")
ax[0].axis('off')

# 이미지 위에 CAM 얹은 시각화
visualization = show_cam_on_image(input_image, grayscale_cam, use_rgb=True)
ax[1].imshow(visualization)
ax[1].set_title("CAM overlaid on image")
ax[1].axis('off')

plt.suptitle(f"Grad-CAM++ for class index: {target_class}", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# 필요 라이브러리 임포트
from ultralytics import YOLO
from pytorch_grad_cam import ScoreCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ✅ 1. YOLO에 맞춘 Custom Target 클래스 정의
class YoloBoxScoreTarget:
    def __init__(self, class_idx):
        self.class_idx = class_idx

    def __call__(self, model_output):
        """
        model_output: (B, A, H, W, 5 + num_classes)
        class_scores: (B, A, H, W)
        → 모든 앵커와 위치에 대해 지정된 class score 평균 계산
        """
        class_scores = model_output[..., 5 + self.class_idx]
        return class_scores.mean()

# ✅ 2. 입력 이미지 로드 및 전처리
img_path = '/home/smile/work/pbs/api/1812741.jpg'
img = cv2.imread(img_path)
img = cv2.resize(img, (384, 384))
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
input_image = img_rgb.astype(np.float32) / 255.0
img_tensor = torch.from_numpy(input_image.transpose(2, 0, 1)).unsqueeze(0).cuda()
img_tensor.requires_grad = False  # Score-CAM은 gradient 안 씀

# ✅ 3. YOLOv10 모델 로딩
model = YOLO('/home/smile/work/pbs/model/yolov5/runsbackup_v10/250403_U2L_001_v10/blood_cell2/weights/best.pt')
model = model.cuda()
model.eval()

# ✅ 4. 내부 PyTorch 모델 구조 접근
internal_model = model.model  # 실제 nn.Module
target_layers = [internal_model.model[13]]  # 예: Neck 끝의 C3 블록

# ✅ 5. Score-CAM 인스턴스 생성
# cam = ScoreCAM(model=internal_model, target_layers=target_layers, use_cuda=True)
cam = ScoreCAM(model=internal_model, target_layers=target_layers)
# ✅ 6. 타겟 클래스 지정 후 Score-CAM 실행
target_class = 9  # 예: RBC 클래스
targets = [YoloBoxScoreTarget(class_idx=target_class)]

grayscale_cam = cam(input_tensor=img_tensor, targets=targets)[0]  # (H, W)

# ✅ 7. 시각화
visualization = show_cam_on_image(input_image, grayscale_cam, use_rgb=True)
plt.imshow(visualization)
plt.axis('off')
plt.title(f"Score-CAM for class index: {target_class}")
plt.show()

In [ ]:
print(internal_model(img_tensor).shape)

In [ ]:
outputs = internal_model(img_tensor)
for i, out in enumerate(outputs):
    print(f"Output {i}: shape = {getattr(out, 'shape', 'not a tensor')}")

In [ ]:
target_class = 0  # 예: RBC 클래스
targets = [YoloBoxScoreTarget(class_idx=target_class)]

grayscale_cam = cam(input_tensor=img_tensor, targets=targets)[0]

In [ ]:
grayscale_cam

In [ ]:
from ultralytics import YOLO
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 이미지 경로
img_path = '/home/smile/work/pbs/api/1812741.jpg'

# 이미지 로딩 및 전처리
img = cv2.imread(img_path)
img = cv2.resize(img, (384, 384))
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
input_image = img_rgb.astype(np.float32) / 255.0
img_tensor = torch.from_numpy(input_image.transpose(2, 0, 1)).unsqueeze(0).cuda()
img_tensor.requires_grad = True  # ✅ Grad-CAM을 위한 gradient 추적 활성화

# YOLO 모델 로드
model = YOLO('/home/smile/work/pbs/model/yolov5/runsbackup_v10/250403_U2L_001_v10/blood_cell2/weights/best.pt')
model = model.cuda()
model.eval()

# 내부 PyTorch 모델 접근
internal_model = model.model

# Grad-CAM 대상 레이어 설정
target_layers = [internal_model.model[-2]]

# Grad-CAM 객체 생성
cam = GradCAM(model=internal_model, target_layers=target_layers)

# 예측 (직접 internal_model 사용해서 gradient 추적 가능하게)
with torch.enable_grad():  # ✅ no_grad 해제
    outputs = internal_model(img_tensor)  # Raw outputs (not decoded)

# YOLO 결과 추론 (클래스 추출 위해 다시 호출)
results = model(img_tensor, verbose=False)[0]
boxes = results.boxes
top_idx = boxes.conf.argmax()
target_class = int(boxes.cls[top_idx].item())

# Grad-CAM 실행
targets = [ClassifierOutputTarget(target_class)]
grayscale_cam = cam(input_tensor=img_tensor, targets=targets)[0]

# 시각화
visualization = show_cam_on_image(input_image, grayscale_cam, use_rgb=True)

# 출력
plt.imshow(visualization)
plt.axis('off')
plt.show()


In [ ]:
import yaml

# YAML 파일 생성
new_data_yaml = {
    'train': '/home/smile/work/pbs06/train/images',
    'val': '/home/smile/work/pbs06/val/images',
    'test': '/home/smile/work/pbs06/test/images',
    'nc': 28,
    'names': [
        "PLT", "clump-PLT", "GiantPlt", "RBC", "Echinocyte", "Elliptocyte", 
        "TearDropCell", "TargetCell", "Schistocyte", "Nucleated", "Reticulocyte",
        "SegNeutrophil", "Lymphocyte", "Monocyte", "Eosinophil", "Basophil", 
        "BandNeutrophil", "Blast", "ToxicGranule", "Myelocyte", "ToxicVacuole", 
        "hyperSeg.", "degeneWBC", "Smudge", "Granule-Lymp.", "Reactive-Lymp.", 
        "Abnormal-Lymp.", "Stomatocyte"
    ]
}

# YAML 파일 경로
yaml_file_path = '/home/smile/work/pbs06/yolov5/data/new_blood_cell.yaml'

# 파일 저장
with open(yaml_file_path, 'w') as f:
    yaml.dump(new_data_yaml, f)

print(f"YAML 파일이 {yaml_file_path}에 저장되었습니다.")

from ultralytics import YOLO

# 모델 로드
best_model = YOLO('/home/smile/work/pbs/model/yolov5/runsbackup_v10/250110_U2L_001_v10/blood_cell/weights/best.pt')

# 모델 평가 수행
test_results = best_model.val(data=yaml_file_path, split='test',conf = 0.75)
# 모델 평가 수행
# test_results = best_model.val(data=yaml_file_path, split='train')

# 평가 결과 출력
print(test_results)

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import FasterRCNNBoxScoreTarget
import torch
import cv2
import numpy as np

# 예시 이미지 불러오기
img_path = '/home/smile/work/pbs06/test/images/sample.jpg'
img = cv2.imread(img_path)
img = cv2.resize(img, (384, 384))
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img_tensor = torch.from_numpy(img_rgb.transpose(2, 0, 1)).float().div(255.0).unsqueeze(0).cuda()

# 모델 로드
model = YOLO('best.pt')
model = model.cuda()
model.eval()

# 내부 모델 구조 접근
internal_model = model.model  # 실제 PyTorch Module

# Grad-CAM 타겟 레이어 설정
target_layers = [internal_model.model[-2]]  # Neck의 마지막 Conv layer (예시)

# CAM 인스턴스 생성
cam = GradCAM(model=internal_model, target_layers=target_layers, use_cuda=True)

# 예측 수행
results = model(img_tensor)[0]  # List[DetectionOutput]

# 가장 높은 confidence의 첫 번째 box 선택
boxes = results.boxes
top_idx = boxes.conf.argmax()
target_class = int(boxes.cls[top_idx].item())

# Grad-CAM 수행
grayscale_cam = cam(input_tensor=img_tensor, targets=[FasterRCNNBoxScoreTarget(target_class=target_class)])[0]
visualization = show_cam_on_image(img_rgb.astype(np.float32) / 255., grayscale_cam, use_rgb=True)

# 시각화 출력
plt.imshow(visualization)
plt.axis('off')
plt.show()

In [ ]:
import torch
from ultralytics import YOLO

# 학습된 모델의 최적 파라미터를 로드
best_model = YOLO('/home/smile/work/pbs/model/yolov5/runsbackup_v10/241107_U2L_001_v10/blood_cell/weights/best.pt')

# 기존 데이터셋 YAML 파일 사용 (테스트 데이터 경로가 이 YAML 파일에 정의되어 있어야 합니다)
data_yaml_path = params['data']  # 이전에 사용한 YAML 파일 경로

# 테스트 데이터셋에 대해 모델 평가 수행 (confidence threshold를 0.75로 설정)
test_results = best_model.val(data=data_yaml_path, split='test')

# 평가 결과 출력
print(test_results)